<a target="_blank" href="https://colab.research.google.com/github/Tensor-Reloaded/Neural-Networks-Template-2025/blob/main/Lab02/Assignment1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Assignment 1 (10 points)**

## **Solving a linear system in python**

In this homework, you will familiarize yourself with key linear algebra con-
cepts and Python programming by solving a system of linear equations. You
will explore multiple methods for solving such systems, including Cramer’s rule
and matrix inversion. By the end of this assignment, you will have a good un-
derstanding of how to represent and manipulate matrices and vectors in Python.

We begin with the following system of 3 linear equations with 3 unknowns:
$$ 2x + 3y - z = 5 $$
$$ x - y + 4z = 6 $$
$$ 3x + y + 2z = 7 $$

This system can be vectorized in the following form:
$$ A \cdot X = B $$
where:
$$
A = \begin{bmatrix}
2 & 3 & -1 \\
1 & -1 & 4 \\
3 & 1 & 2
\end{bmatrix}, \quad 
X = \begin{bmatrix}
x \\
y \\
z
\end{bmatrix}, \quad 
B = \begin{bmatrix}
5 \\
6 \\
7
\end{bmatrix}
$$

**Considerations**
- do not use any linear algebra framework such as $numpy$
- use python lists as data structures for matrices and vectors
- experiment with other values for the coefficients and free terms

### **1. Parsing the System of Equations (1 point)**

The first task is to implement a Python script that reads a system of linear equations from a text file and parses it into a matrix $A$ and a vector $B$. You will use the input format described below to extract the coefficients for $A$ and $B$.

**Input File Format**
```text
2x + 3y - z = 5
x - y + 4z = 6
3x + y + 2z = 7
```

Note that the coefficients are always in the order x, y and z and the terms are always space separated

In [12]:
import pathlib

def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    A = []
    B = []

    with open(path, 'r') as file:
        for line in file:
            line = line.strip()
                
            parts = line.split('=')
            left_side = parts[0].replace(' ', '')
            right_side = float(parts[1].replace(' ', ''))
            
            def extract_coef(var: str) -> float:
                idx = left_side.find(var)
                if idx == -1: 
                    return 0.0

                #obtinem semnul
                start = idx - 1
                while start >= 0 and left_side[start] not in '+-':
                    start -= 1
                
                coef_str = left_side[start+1:idx]
                sign = -1.0 if start >= 0 and left_side[start] == '-' else 1.0
                
                #x sau -x
                if coef_str == '':
                    return 1.0 * sign
                return float(coef_str) * sign
            
            A.append([extract_coef('x'), extract_coef('y'), extract_coef('z')])
            B.append(right_side)
            
    return A, B

A, B = load_system(pathlib.Path("sistem.txt"))

### **2. Matrix and Vector Operations (5 points)**

Once you have successfully parsed the matrix and vector, complete the following exercises to manipulate and understand basic matrix and vector operations. Write Python functions for each of these tasks:

#### 2.1. Determinant

Write a function to compute the determinant of matrix $A$. Recall one of the formulae for the determinant of a $3x3$ matrix:
$$ \text{det}(A) = a_{11}(a_{22}a_{33} - a_{23}a_{32}) - a_{12}(a_{21}a_{33} - a_{23}a_{31}) + a_{13}(a_{21}a_{32} - a_{22}a_{31}) $$

In [9]:
def determinant(matrix: list[list[float]]) -> float:
    a11, a12, a13 = matrix[0]
    a21, a22, a23 = matrix[1]
    a31, a32, a33 = matrix[2]
    
    t1 = a11 * (a22 * a33 - a23 * a32)
    t2 = a12 * (a21 * a33 - a23 * a31)
    t3 = a13 * (a21 * a32 - a22 * a31)
    
    return t1 - t2 + t3

print(f"{determinant(A)=}")

determinant(A)=14.0


#### 2.2. Trace

Compute the sum of the elements along the main diagonal of matrix $A$. For a matrix $A$, this is:
$$ \text{Trace}(A) = a_{11} + a_{22} + a_{33} $$

In [10]:
def trace(matrix: list[list[float]]) -> float:
    return matrix[0][0] + matrix[1][1] + matrix[2][2]

print(f"{trace(A)=}")

trace(A)=3.0


#### 2.3. Vector norm

Compute the Euclidean norm of vector $B$, which is:
$$ ||B|| = \sqrt{b_1^2 + b_2^2 + b_3^2} $$

In [13]:
def norm(vector: list[float]) -> float:
    suma = vector[0]**2 + vector[1]**2 + vector[2]**2
    return suma ** 0.5

print(f"{norm(B)=}")

norm(B)=10.488088481701515


#### 2.4. Transpose of matrix

Write a function to compute the transpose of matrix $A$. The transpose of a matrix $A$ is obtained by swapping its rows and columns.
    

In [16]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    M = []
    for i in range(3):
        row = []
        for j in range(3):
            row.append(matrix[j][i])

        M.append(row)
        
    return M

print(f"{transpose(A)=}")

transpose(A)=[[2.0, 1.0, 3.0], [3.0, -1.0, 1.0], [-1.0, 4.0, 2.0]]


#### 2.5. Matrix-vector multiplication

Write a function that multiplies matrix $A$ with vector $B$.

In [18]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    res = []

    for i in range((len(matrix))):
        row_sum = (matrix[i][0] * vector[0] + 
                   matrix[i][1] * vector[1] + 
                   matrix[i][2] * vector[2])
        res.append(row_sum)
    return res

print(f"{multiply(A, B)=}")

multiply(A, B)=[21.0, 27.0, 35.0]


### **3. Solving using Cramer's Rule (1 point)**

Now that you have explored basic matrix operations, solve the system of linear equations using Cramer's rule.

**Cramer's Rule:**

Cramer's rule allows you to solve for each unknown $x$, $y$, and $z$ using determinants. For example:
$$ x = \frac{\text{det}(A_x)}{\text{det}(A)}, \quad y = \frac{\text{det}(A_y)}{\text{det}(A)}, \quad z = \frac{\text{det}(A_z)}{\text{det}(A)} $$
where $A_x$, $A_y$, and $A_z$ are matrices formed by replacing the respective column of matrix $A$ with vector $B$.

In [19]:
def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float]:
    det_A = determinant(matrix)
    
    if det_A == 0:
        return [] 
        
    def swap_column(matrice_veche, col, vec):
        matrice_noua = []
        for i in range(len(matrice_veche)):
            rand_nou = list(matrice_veche[i])
            rand_nou[col] = vec[i]
            matrice_noua.append(rand_nou)
        return matrice_noua

    A_x = swap_column(matrix, 0, vector)
    A_y = swap_column(matrix, 1, vector)
    A_z = swap_column(matrix, 2, vector)
    
    det_Ax = determinant(A_x)
    det_Ay = determinant(A_y)
    det_Az = determinant(A_z)

    x = det_Ax / det_A
    y = det_Ay / det_A
    z = det_Az / det_A
    
    return [x, y, z]

print(f"{solve_cramer(A, B)=}")

solve_cramer(A, B)=[0.35714285714285715, 2.0714285714285716, 1.9285714285714286]


### **4. Solving using Inversion (3 points)**

Finally, solve the system by computing the inverse of matrix $A$ and multiplying it by vector $B$.
$$ A \cdot X = B \rightarrow X = A^{-1} \cdot B $$
**Adjugate Method for Matrix Inversion:**

To find the inverse of matrix $ A $, you can use the adjugate method:
$$ A^{-1} = \frac{1}{\text{det}(A)} \times \text{adj}(A) $$
where $\text{adj}(A)$ is the adjugate (or adjoint) matrix, which is the transpose of the cofactor matrix of $ A $.

**Cofactor Matrix:**

The cofactor matrix is a matrix where each element is replaced by its cofactor. The cofactor of an element $a_{ij}$ is given by:
$$ (-1)^{i+j} \times \text{det}(M_{ij}) $$
where $M_{ij}$ is the minor of element $a_{ij}$, which is the matrix obtained by removing the $i$-th row and $j$-th column from matrix $A$.

In [22]:
def minor(matrix: list[list[float]], i: int, j: int) -> list[list[float]]:
    minor = []
    
    for row in range(len(matrix)):
        if row == i:
            continue
            
        new_row = []
        for col in range(len(matrix[0])):
            if col == j:
                continue
            new_row.append(matrix[row][col])
            
        minor.append(new_row)
        
    return minor

def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    cofactor = []
    
    for i in range(len(matrix)):
        row = []
        for j in range(len(matrix[0])):
            # Extragem minorul pentru poziția curentă (i, j)
            m = minor(matrix, i, j)
            det = m[0][0] * m[1][1] - m[0][1] * m[1][0]
            semn = (-1) ** (i + j)
            
            row.append(semn * det)
            
        cofactor.append(row)
        
    return cofactor

def adjoint(matrix: list[list[float]]) -> list[list[float]]:
    cof = cofactor(matrix)
    adj = []

    for i in range(len(cof[0])):
        row = []
        for j in range(len(cof)):
            row.append(cof[j][i])
        adj.append(row)
        
    return adj

def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    det_A = determinant(matrix)
    
    if det_A == 0:
        return []
        
    adj_A = adjoint(matrix)

    inv_A = []
    for i in range(len(adj_A)):
        row = []
        for j in range(len(adj_A[0])):
            row.append(adj_A[i][j] / det_A)
        inv_A.append(row)

    rezultat = multiply(inv_A, vector)
    
    return rezultat

print(f"{solve(A, B)=}")

solve(A, B)=[0.35714285714285765, 2.071428571428571, 1.9285714285714288]
